In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

!pip install category_encoders
!pip install optuna
from category_encoders import TargetEncoder
import optuna
import lightgbm as lgb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 37.1 MB/s eta 0:00:00


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sub = pd.read_csv('sample_submission.csv')

In [ ]:
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022.0,0.0,50.0,2.0,39.0,8.0,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025.0,1.0,27.0,2.0,7.0,4.0,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022.0,0.0,59.0,3.0,22.0,13.0,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023.0,0.0,2.0,1.0,2.0,7.0,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022.0,1.0,26.0,3.0,6.0,2.0,107.878,8.965,-14.139,0.361111,3.0,0.0


In [ ]:
train.isnull().sum()

,0
id,0
Driver,1
Compound,1
Race,1
Year,1
PitStop,1
LapNumber,1
Stint,1
TyreLife,1
Position,1


In [ ]:
print('Compound ::',train['Compound'].unique())
print('Race ::' , train['Race'].unique())

Compound :: ['HARD' 'MEDIUM' 'INTERMEDIATE' 'SOFT' 'WET' nan]
Race :: ['Canadian Grand Prix' 'Dutch Grand Prix' 'Austrian Grand Prix'
 'Pre-Season Testing' 'Azerbaijan Grand Prix' 'Saudi Arabian Grand Prix'
 'Belgian Grand Prix' 'United States Grand Prix' 'Italian Grand Prix'
 'Hungarian Grand Prix' 'Japanese Grand Prix' 'São Paulo Grand Prix'
 'Bahrain Grand Prix' 'Las Vegas Grand Prix' 'Monaco Grand Prix'
 'British Grand Prix' 'Australian Grand Prix' 'Spanish Grand Prix'
 'Miami Grand Prix' 'French Grand Prix' 'Abu Dhabi Grand Prix'
 'Chinese Grand Prix' 'Mexico City Grand Prix' 'Emilia Romagna Grand Prix'
 'Singapore Grand Prix' 'Qatar Grand Prix' nan]


In [ ]:
numaric_col = train.select_dtypes(exclude = ['object']).columns
object_col = test.select_dtypes(include = ['object']).columns
print("Numaric Col :",numaric_col)
print("Object Col :" , object_col)

Numaric Col : Index(['id', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change', 'PitNextLap'],
      dtype='object')
Object Col : Index(['Driver', 'Compound', 'Race'], dtype='object')


In [ ]:
for i,col in enumerate(numaric_col):
    print(col , ':' ,train[col].skew())

id : 3.461024476137005e-05
Year : 0.0025201807439209226
PitStop : 2.122085210560281
LapNumber : 0.6405306486150917
Stint : 1.161958636405021
TyreLife : 1.0281472023489693
Position : 0.07469407228140501
LapTime (s) : 0.834708980361538
LapTime_Delta : -55.222388659869324
Cumulative_Degradation : -0.3286290107263015
RaceProgress : 0.6948408784666479
Position_Change : -0.09950199577822844
PitNextLap : 1.507552225039876


In [ ]:
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022.0,0.0,50.0,2.0,39.0,8.0,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025.0,1.0,27.0,2.0,7.0,4.0,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022.0,0.0,59.0,3.0,22.0,13.0,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023.0,0.0,2.0,1.0,2.0,7.0,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022.0,1.0,26.0,3.0,6.0,2.0,107.878,8.965,-14.139,0.361111,3.0,0.0


In [ ]:
train.drop(['id'], axis=1, inplace=True)

In [ ]:
train.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,D109,HARD,Canadian Grand Prix,2022.0,0.0,50.0,2.0,39.0,8.0,78.491,-7.564,21.019,0.714286,5.0,1.0
1,D086,HARD,Dutch Grand Prix,2025.0,1.0,27.0,2.0,7.0,4.0,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,ZON,HARD,Austrian Grand Prix,2022.0,0.0,59.0,3.0,22.0,13.0,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,SPE,MEDIUM,Pre-Season Testing,2023.0,0.0,2.0,1.0,2.0,7.0,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,D019,HARD,Azerbaijan Grand Prix,2022.0,1.0,26.0,3.0,6.0,2.0,107.878,8.965,-14.139,0.361111,3.0,0.0


In [ ]:
train['PitNextLap'].value_counts()

,count
PitNextLap,
0.0,34635
1.0,8607


In [ ]:
def data_encode(df):
    df = df.copy()

    df = pd.get_dummies(df,columns=['Compound'],drop_first=False , dtype=int)

    return df

groups = train['Driver'].astype(str) + "_" + train['Race'].astype(str)

train = data_encode(train)

In [ ]:
train.head()

,Driver,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,D109,Canadian Grand Prix,2022.0,0.0,50.0,2.0,39.0,8.0,78.491,-7.564,21.019,0.714286,5.0,1.0,1,0,0,0,0
1,D086,Dutch Grand Prix,2025.0,1.0,27.0,2.0,7.0,4.0,75.095,-32.617,-223.207,0.346154,-3.0,0.0,1,0,0,0,0
2,ZON,Austrian Grand Prix,2022.0,0.0,59.0,3.0,22.0,13.0,70.945,-7.540,-100.529,0.819444,3.0,1.0,1,0,0,0,0
3,SPE,Pre-Season Testing,2023.0,0.0,2.0,1.0,2.0,7.0,94.361,-7.324,-7.324,0.076923,0.0,0.0,0,0,1,0,0
4,D019,Azerbaijan Grand Prix,2022.0,1.0,26.0,3.0,6.0,2.0,107.878,8.965,-14.139,0.361111,3.0,0.0,1,0,0,0,0


In [ ]:
def drop_cols(df):
    df = df.copy()
    df = df.drop(columns = ['RaceProgress'])
    return df

In [ ]:
train.head()

,Driver,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET
0,D109,Canadian Grand Prix,2022.0,0.0,50.0,2.0,39.0,8.0,78.491,-7.564,21.019,0.714286,5.0,1.0,1,0,0,0,0
1,D086,Dutch Grand Prix,2025.0,1.0,27.0,2.0,7.0,4.0,75.095,-32.617,-223.207,0.346154,-3.0,0.0,1,0,0,0,0
2,ZON,Austrian Grand Prix,2022.0,0.0,59.0,3.0,22.0,13.0,70.945,-7.540,-100.529,0.819444,3.0,1.0,1,0,0,0,0
3,SPE,Pre-Season Testing,2023.0,0.0,2.0,1.0,2.0,7.0,94.361,-7.324,-7.324,0.076923,0.0,0.0,0,0,1,0,0
4,D019,Azerbaijan Grand Prix,2022.0,1.0,26.0,3.0,6.0,2.0,107.878,8.965,-14.139,0.361111,3.0,0.0,1,0,0,0,0


In [ ]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43243 entries, 0 to 43242
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Driver                  43242 non-null  object 
 1   Race                    43242 non-null  object 
 2   Year                    43242 non-null  float64
 3   PitStop                 43242 non-null  float64
 4   LapNumber               43242 non-null  float64
 5   Stint                   43242 non-null  float64
 6   TyreLife                43242 non-null  float64
 7   Position                43242 non-null  float64
 8   LapTime (s)             43242 non-null  float64
 9   LapTime_Delta           43242 non-null  float64
 10  Cumulative_Degradation  43242 non-null  float64
 11  RaceProgress            43242 non-null  float64
 12  Position_Change         43242 non-null  float64
 13  PitNextLap              43242 non-null  float64
 14  Compound_HARD           43243 non-null

In [ ]:
train.nunique()

,0
Driver,702
Race,26
Year,4
PitStop,2
LapNumber,77
Stint,8
TyreLife,75
Position,20
LapTime (s),17317
LapTime_Delta,18459


In [ ]:
driver_freq = train["Driver"].value_counts(normalize=True)
train["Driver_FE"] = train["Driver"].map(driver_freq)

race_freq = train["Race"].value_counts(normalize=True)
train["Race_FE"] = train["Race"].map(race_freq)

train.drop(columns=["Driver", "Race"], inplace=True)

In [ ]:
train

,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap,Compound_HARD,Compound_INTERMEDIATE,Compound_MEDIUM,Compound_SOFT,Compound_WET,Driver_FE,Race_FE
0,2022.0,0.0,50.0,2.0,39.0,8.0,78.491,-7.564,21.019,0.714286,5.0,1.0,1,0,0,0,0,0.002058,0.048471
1,2025.0,1.0,27.0,2.0,7.0,4.0,75.095,-32.617,-223.207,0.346154,-3.0,0.0,1,0,0,0,0,0.002428,0.056797
2,2022.0,0.0,59.0,3.0,22.0,13.0,70.945,-7.540,-100.529,0.819444,3.0,1.0,1,0,0,0,0,0.003538,0.048171
3,2023.0,0.0,2.0,1.0,2.0,7.0,94.361,-7.324,-7.324,0.076923,0.0,0.0,0,0,1,0,0,0.003399,0.050275
4,2022.0,1.0,26.0,3.0,6.0,2.0,107.878,8.965,-14.139,0.361111,3.0,0.0,1,0,0,0,0,0.003446,0.028491
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43238,2024.0,0.0,4.0,2.0,6.0,19.0,82.970,-12.790,-12.907,0.051282,-16.0,1.0,0,1,0,0,0,0.003330,0.048471
43239,2023.0,0.0,20.0,1.0,20.0,9.0,93.104,-0.528,-12.069,0.263158,0.0,0.0,1,0,0,0,0,0.001272,0.043083
43240,2022.0,0.0,6.0,1.0,6.0,8.0,104.914,19.823,-31.746,0.083333,2.0,0.0,0,0,1,0,0,0.002058,0.050275
43241,2025.0,0.0,12.0,1.0,12.0,20.0,77.516,-6.088,-103.753,0.155844,-10.0,0.0,0,0,1,0,0,0.001850,0.056797


In [ ]:
!pip install catboost

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    VotingClassifier,
    StackingClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.linear_model import LogisticRegression

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier







import optuna

TARGET = "PitNextLap"

train.dropna(subset=[TARGET], inplace=True)

X = train.drop(columns=[TARGET])
y = train[TARGET]


# TRAIN VALID SPLIT

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train Shape:", X_train.shape)
print("Validation Shape:", X_valid.shape)


#STRATIFIED K FOLD


skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)



# BASELINE CATBOOST MODEL


cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    eval_metric='AUC',
    loss_function='Logloss',
    verbose=200,
    random_seed=42
)

cat_model.fit(
    X_train,
    y_train,
    eval_set=(X_valid, y_valid),
    early_stopping_rounds=100,
    use_best_model=True
)



# PREDICTIONS & EVALUATION

train_preds = cat_model.predict_proba(X_train)[:, 1]
valid_preds = cat_model.predict_proba(X_valid)[:, 1]

train_auc = roc_auc_score(y_train, train_preds)
valid_auc = roc_auc_score(y_valid, valid_preds)

print("\n----------------------")
print("BASELINE CATBOOST RESULTS")
print("-------------------------")
print(f"Train ROC-AUC : {train_auc:.5f}")
print(f"Valid ROC-AUC : {valid_auc:.5f}")


# OPTUNA TUNING (Automatic Hyperparameter tuning opensource library)

print("\nStarting Optuna Hyperparameter Tuning...")

def objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'loss_function': 'Logloss',
        'eval_metric': 'AUC',
        'verbose': 0,
        'random_seed': 42
    }

    model = CatBoostClassifier(**params)

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=skf,
        scoring='roc_auc',
        n_jobs=-1
    )
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print("\nBEST OPTUNA PARAMETERS:", study.best_params)


# FINAL ENSEMBLE (SIMPLIFIED)

best_catboost = CatBoostClassifier(**study.best_params, verbose=0, random_seed=42)
best_catboost.fit(X_train, y_train)

print("\nPipeline execution finished.")

Train Shape: (34593, 18)
Validation Shape: (8649, 18)
0:	test: 0.8940706	best: 0.8940706 (0)	total: 63.3ms	remaining: 1m 3s
200:	test: 0.9306701	best: 0.9306701 (200)	total: 3s	remaining: 11.9s
400:	test: 0.9341258	best: 0.9341258 (400)	total: 5.92s	remaining: 8.84s
600:	test: 0.9349593	best: 0.9350307 (578)	total: 8.86s	remaining: 5.88s


[I 2026-05-24 12:12:04,798] A new study created in memory with name: no-name-737662b9-3b01-44c3-b5df-ff911a6636a2


Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.9352806026
bestIteration = 678

Shrink model to first 679 iterations.

BASELINE CATBOOST RESULTS
Train ROC-AUC : 0.98425
Valid ROC-AUC : 0.93528

Starting Optuna Hyperparameter Tuning...


[I 2026-05-24 12:12:49,665] Trial 0 finished with value: 0.9357880616862115 and parameters: {'iterations': 839, 'learning_rate': 0.06957775153743635, 'depth': 7, 'l2_leaf_reg': 9.238158513820615}. Best is trial 0 with value: 0.9357880616862115.
[I 2026-05-24 12:13:37,413] Trial 1 finished with value: 0.9354813913962022 and parameters: {'iterations': 1303, 'learning_rate': 0.08022407410237144, 'depth': 5, 'l2_leaf_reg': 3.2954449719967203}. Best is trial 0 with value: 0.9357880616862115.
[I 2026-05-24 12:15:56,298] Trial 2 finished with value: 0.9344650745063209 and parameters: {'iterations': 1394, 'learning_rate': 0.04277074819542437, 'depth': 9, 'l2_leaf_reg': 7.005203889744014}. Best is trial 0 with value: 0.9357880616862115.
[I 2026-05-24 12:16:27,474] Trial 3 finished with value: 0.9291348612741176 and parameters: {'iterations': 873, 'learning_rate': 0.015375382947320696, 'depth': 5, 'l2_leaf_reg': 4.3553432121991955}. Best is trial 0 with value: 0.9357880616862115.
[I 2026-05-24 1


BEST OPTUNA PARAMETERS: {'iterations': 697, 'learning_rate': 0.05468061116385157, 'depth': 6, 'l2_leaf_reg': 3.3040072471834114}

Pipeline execution finished.


In [ ]:
TARGET = "PitNextLap"
X = train.drop(columns=[TARGET])


# PREPROCESS TEST DATA & GENERATE SUBMISSION

# 1. Prepare a copy and keep IDs
test_df = test.copy()
test_ids = test_df['id']

# 2. One-Hot Encoding
test_df = pd.get_dummies(test_df, columns=['Compound'], drop_first=False, dtype=int)

# 3. Frequency Encoding using training mappings
test_df['Driver_FE'] = test_df['Driver'].map(driver_freq)
test_df['Race_FE'] = test_df['Race'].map(race_freq)

# 4. fill NaN with 0
test_df['Driver_FE'] = test_df['Driver_FE'].fillna(0)
test_df['Race_FE'] = test_df['Race_FE'].fillna(0)

# 5. Drop columns not used as features
test_df.drop(columns=['id', 'Driver', 'Race'], inplace=True)

# 6. Align columns with the training set X
# Add missing columns (like specific Compound types) and remove extra ones
missing_cols = set(X.columns) - set(test_df.columns)
for c in missing_cols:
    test_df[c] = 0
test_df = test_df[X.columns]

# 7. Generate Predictions
print("Generating predictions...")
test_probs = best_catboost.predict_proba(test_df)[:, 1]

# 8. Create Submission CSV
submission = pd.DataFrame({
    'id': test_ids,
    'PitNextLap': test_probs
})

submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' saved.")
submission.head()

Generating predictions...
Submission file 'submission.csv' saved.


,id,PitNextLap
0,439140,0.011150
1,439141,0.006851
2,439142,0.008313
3,439143,0.055547
4,439144,0.748228
